# Lab1. Part 2: infographic of relationships between  characters in the Game of Thrones

**Brief description**

[Game of Thrones](https://en.wikipedia.org/wiki/Game_of_Thrones) is arguably one of the biggest pop culture phenomena to hit the public zeitgeist in the last decade.

In this **Part 2 of the Lab 1**, we will need to analyze data on relationships (family ties) between characters in the Game of Thrones  and then build a network (a set of graphs representing the main Houses - Dynasties).




The *game-of-thrones-characters-groups.json* dataset stores about characters and their relationships (family ties).

We will build and visualize a undirected graph where nodes are characters (the central node will be the House - Dynasty name) and and edges represent dynasty membership.

Let's imagine that we need to summerize relationships between various characters in the Game of Thrones.

We have *game-of-thrones-characters-groups.json* dataset that captures data about character relationships.

In [ ]:
import json
import io
import os

In [ ]:
file_name = "data/game-of-thrones-characters-groups.json"
path="data"

In [ ]:
json_files = [os.path.join(root, name) 
              for root, dirs, files in os.walk(path) 
              for name in files 
              if name.endswith((".json"))] #If we needed to read several files extensions: if name.endswith((".ext1", ".ext2"))

print('Number of JSON files ready to be loaded: ' + str(len(json_files)))

In [ ]:
print('Path to the first file: '+json_files[0])

In [ ]:
#Open the file using the name of the json file witn open() function
#Read the json file using load() and put the json data into a variable.
with open(json_files[0]) as f:
   json_data = json.load(f)

In [ ]:
json_data

## Task 1.

1.1 Create **Dynasty Class** by using the template below:

In [ ]:
class Dynasty:
    def __init__(self,name):
        self._name=name # House name, for.ex."Martell"
        self.characters=[] # Family members ("Doran Martell","Ellaria Sand","Nymeria Sand",...)


    @property
    def name(self): # getter for the private instance attribute _name
        return self._name

    @name.setter
    def name(self, value):
        if value=="":
            raise ValueError('Name is empty')

        self._name = value

    def append(self, ch): # to append character to the House (during reading data from JSON-file)
         if type(ch) is not str:
             raise TypeError('Ch is not of type str')

         self.characters.append(ch)

    def __iter__(self): # to loop throw the list of characters via IN operator (for ex. for person in house: ....)
         for ch in self.characters:
             yield ch

    def __contains__(self, ch): # to check if the character belongs to the house (for ex., if person in house ...)
        for e in ch:
            if e == ch:
                return True
        return False

    def __str__(self): # to print like print(house) - > displat the house's name
        return f"This is the house of {self._name}!"
    
    def getStrength(self): # return N of family members in this house (int)
        return len(self.characters)

1.2 Use the code below to check that your Dynasty class meets all requirement

In [ ]:
for data in json_data['groups']:
    house = Dynasty(data['name'])
    for character in data['characters']:
        house.append(character)
    print(house)
    print("Our members:")
    for person in house:
        print(person)
    print(f"We have {house.getStrength()} family members!!!")


## Task 2.

2.1 Create **GameOfThronesGraph Class** to store data abot all Houses by using the temlate below:

In [ ]:
class GameOfThronesGraph:
    def __init__(self, corpus):
        #initialisation of dictionary that will store all houses. They keys are Houses' (Dynasty) names, the values are Dynasty objects.
        self.houses = {}
        #Load the house corpus
        for data_item in corpus:
            dynasty = Dynasty(data_item['name'])
            for character in data_item['characters']:
                dynasty.append(character)
            self.houses[data_item['name']] = dynasty


    def __iter__(self):  # for the case like the following: for house in GameOfThronesHouses:
        for e in self.houses.keys():
            yield self.houses[e]

    def __contains__(self, h):  #Check if h (house's name) is a key in dict houses - the house is in the graph
        return h in self.houses.keys()

In [ ]:
corpusData=json_data['groups']

In [ ]:
GameOfThronesHouses=GameOfThronesGraph(corpusData)

In [ ]:
for house in GameOfThronesHouses:
    print(house)

2.2 Let's add a visualization to show the power level of each dynasty.
Run the code below.

In [ ]:
visualisationData={}
legendData=[]
for house in GameOfThronesHouses:
  print(house)
  print(f"Strength: {house.getStrength()}")
  visualisationData[house.name]=house.getStrength()
  legendData.append(house.name)

In [ ]:
visualisationData

In [ ]:
legendData

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
#Configure your x and y values from the dictionary:
x= list(visualisationData.keys())
y=list(visualisationData.values())

#Create the graph = create seaborn barplot
ax=sns.barplot(x=x,y=y)

#specfiy axis labels
ax.legend(legendData)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1.05, 1))
ax.set(xlabel='Houses',
       ylabel='Strength (N family members)',
       title='Strength of GameOfThronesHouses')

plt.xticks(rotation=45)
#display barplot
plt.show()

## Task 3.

Now we need to create the graph with a suitable Python library  based on our graph data stored in *GameOfThronesHouses*.

[NetworkX](https://networkx.org/) is a Python package for the creation, manipulation, and study of the structure, dynamics, and functions of complex networks.

In [ ]:
import networkx as nx

In [ ]:
g = nx.Graph() # graph initialization

Add nodes into the Graph (add characters into Game Of Thrones)

Let's use *seaborn.color_palette* and create a dictionary to specify different color for different families.

In [ ]:
import seaborn as sns

N_houses=0
colorKeys=[]
for house in GameOfThronesHouses:
    if house.name!="Include":
        N_houses+=1
        colorKeys.append(house.name)
sns.color_palette("husl", N_houses) # N_houses colors

In [ ]:
list(sns.color_palette("husl", N_houses))

In [ ]:
colorKeys

In [ ]:
nodeColors=dict(zip(colorKeys, [tuple(int(c*255) for c in cs) for cs in sns.color_palette("husl", N_houses)]))
nodeColors

In [ ]:
#this doesn't work for us because Pyvis requires strings (hexadecimal) as values for the color attribute, not RGB tuples. 
# We need to convert RGB tuples to hexadecimal strings later
#for house in GameOfThronesHouses:
    #if house.name!="Include":
        #g.add_node(house.name, size=house.getStrength(), color=nodeColors[house.name])
    

Add nodes to the graph (via *g.add_node()*) with their names and sizes only.

First we need to add main nodes - houses, and add other nodes - family members after that.

In [ ]:
for house in GameOfThronesHouses:
    if house.name!="Include":
        # add the house's name as a node to the graph g (houses's strength values is used as a node's size)
        #your code here
        

In [ ]:
for node, attributes in g.nodes(data=True): # run this code to check your code above
    print(f"Node: {node}, Attributes: {attributes}")

Now we can add nodes - family members from each house

In [ ]:
for house in GameOfThronesHouses:
    if house.name!="Include":
        # add each character as a node to the graph g 
        #your code here
        

In [ ]:
for node, attributes in g.nodes(data=True): # run this code to check your code above
    print(f"Node: {node}, Attributes: {attributes}")

Add edges (*myEdges=[]*):
1. add connections between a House and its family members
2. add connections between members belonging to the same House (not presented in my examples)

In [ ]:
myEdges=[]

In [ ]:
for house in GameOfThronesHouses:
    if house.name!="Include":
        for person in house:
            #your code here

            

In [ ]:
print("Connections between a House and its family members:") # run this code to check your code above
myEdges

In [ ]:
g.add_edges_from(myEdges) # run this code to add edges to our graph g

In [ ]:
list(g.edges)# run this code  to check the edges in our graph g

In [ ]:
len(list(g.edges)) # N of edges =89!!! check yours :)

In [ ]:
from pyvis.network import Network

All networks (graphs) must be instantiated as a *Network* class instance

In [ ]:
GameOfThronesNet = Network(
                bgcolor ="#242020",
                font_color = "white",
                height = "1000px",
                width = "100%",
                notebook=True,
                cdn_resources = "remote")

In [ ]:
# generate the graph
GameOfThronesNet.from_nx(g)  

Specify colors for the Houses and their members by using the code below:

In [ ]:
for node in GameOfThronesNet.nodes:
    if node["id"] in GameOfThronesHouses:
        # Convert RGB to hexadecimal string
        node["color"] = '#%02x%02x%02x' % nodeColors[node["id"]]
    else:
        for house in GameOfThronesHouses: 
            if house.name !="Include":# apple the coloer of the House to this family member
                if node["id"] in house:
                    node["color"] = '#%02x%02x%02x' % nodeColors[house.name]

In [ ]:
GameOfThronesNet.nodes

 Display the network as an interactive HTML file using the *show()* method.

In [ ]:
GameOfThronesNet.show("GameOfThronesNet.html",notebook=False)

![image.png](attachment:image.png)


![image.png](attachment:image.png)

Task 4.

Now you are ready to deploy your visualization App as a **web service**. Let's develop a **Streamlit application** (**lab1app2.py**) and deploy it on Streamlit Community Cloud.

The application includes 3 tabs, each of which displays different data (visualization). See required results below.

![image.png](attachment:image.png)

![image.png](attachment:image.png)

![image.png](attachment:image.png)